
---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

An nPrint CSV represents a network trace as a table in which each row corresponds to one packet in chronological order. The columns represent packet-header fields, such as protocol indicators, flags, addresses, ports, and other metadata. Many fields are encoded using values such as 0, 1, and -1, where -1 generally indicates that a field is not applicable to that packet.

A 2D image places packet fields next to one another horizontally and consecutive packets next to one another vertically. This spatial organization allows a vision model to detect both relationships among fields within one packet and patterns that develop across multiple packets. A CSV contains the same information, but a model processing rows independently may have more difficulty learning both kinds of dependencies simultaneously.

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

In [4]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image

def convert_nprints_to_png(input_dir, output_dir, max_packets=1024):
    os.makedirs(output_dir, exist_ok=True)
    
    file_paths = glob.glob(os.path.join(input_dir, "*.nprint*"))
    print(f"Loading {len(file_paths)} traces from: {input_dir}")
    
    if not file_paths:
        print(f"Error: No .nprint files found in {input_dir}. Please check your path.")
        return

    for file_path in file_paths:
        df = pd.read_csv(file_path, low_memory=False)
        if df.columns[0].lower().startswith("unnamed"):
            df = df.drop(df.columns[0], axis=1)
            
        df_sub = df.iloc[:max_packets, :]
        
        def map_pixel(val):
            s = str(val).strip()
            if s == '-1' or pd.isna(val):
                return 128
            elif s == '1':
                return 255
            return 0

        mapped = df_sub.map(map_pixel)
        img_array = mapped.to_numpy(dtype=np.uint8)
        
        if img_array.shape[0] < max_packets:
            pad_rows = max_packets - img_array.shape[0]
            padding = np.full((pad_rows, img_array.shape[1]), 128, dtype=np.uint8)
            img_array = np.vstack([img_array, padding])
            
        base_name = os.path.basename(file_path).split('.nprint')[0] + '.png'
        out_path = os.path.join(output_dir, base_name)
        Image.fromarray(img_array).save(out_path)
        
    print(f"Done! Successfully converted and saved images to: {output_dir}")

possible_paths = [
    "./nd_data/real_nprints",
    "../nd_data/real_nprints",
    "/content/nd_data/real_nprints"
]

target_input = None
for p in possible_paths:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, "*.nprint*"))) > 0:
        target_input = p
        break

if target_input:
    convert_nprints_to_png(target_input, './nd_data/student_converted_images')
else:
    print("Could not find the real_nprints folder. Confirm the folder was unzipped properly.")

Loading 200 traces from: ../nd_data/real_nprints
Done! Successfully converted and saved images to: ./nd_data/student_converted_images


In [6]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python3 ./scripts/nprint_to_png.py -i ../nd_data/real_nprints/ -o ../nd_data/real_traffic_images

Processing teams_1024_10.nprint
/Users/sandyzheng/Desktop/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitch_1024_7.nprint
/Users/sandyzheng/Desktop/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing netflix_1024_15.nprint
/Users/sandyzheng/Desktop/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitter_1024_17.nprint
/Users/sandyzheng/Desktop/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing n

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

My Q2 method uses simple grayscale pixels (0, 128, 255) to represent fields, while NetDiffusion packs protocol data across 3-channel RGB. The upside of my approach is that it’s crisp and has zero color-bleeding—a bit is either on, off, or not applicable. The downside is that off-the-shelf vision models expect 3-channel RGB, so feeding them single-channel images wastes their pre-trained weights and makes traces take up a lot more visual real estate.

NetDiffusion’s RGB setup works right out of the box with standard Stable Diffusion models and compresses way more header info into fewer pixels. The catch is that diffusion models think in smooth color gradients, not strict binary code. That continuous color drift can easily blend RGB values together, which introduces subtle noise and makes turning the image back into valid packet headers a real headache.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

Looking at the generated images versus the real ones, the synthetic samples look slightly blurry and have soft, anti-aliased transitions instead of sharp, crisp grid lines. The colors also drift into continuous, muddy shades rather than sticking to pure, distinct color values. Sometimes there is even slight spatial warping along the columns. This is a big problem for converting back to nPrint because network headers require exact bits, a fuzzy or slightly off-color pixel can accidentally flip a TCP flag or corrupt an IP address.

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

To convert the images back, I would first snap the image dimensions back to the original grid ($1024 \times \text{columns}$) using nearest-neighbor interpolation so boundaries stay sharp. Next, I'd run a color-quantization step that measures the distance of each pixel to the nearest known reference color centroid, forcing continuous values back to clean discrete bits ($0, 1, -1$). Finally, I’d apply sanity checks on the headers, like ensuring one-hot protocol flags aren't mutually conflicting and valid protocol numbers are kept, before mapping everything back into the standard nPrint columns.

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [9]:
# Step 1: Color Augmentation
!python3 ./scripts/color_processor.py \
  --input_dir="../nd_data/generated_traffic_images" \
  --output_dir="../nd_data/color_corrected_generated_traffic_images"

Processed 100 images.


In [10]:
# Step 2: Image-to-nPrint Conversion
!python3 ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ../nd_data/color_corrected_generated_traffic_images \
  --output_dir ./nd_data/generated_nprint

Processing ../nd_data/color_corrected_generated_traffic_images/teams_5.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_5.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/twitch_3.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_3.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/zoom_7.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_7.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/zoom_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_6.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/twitch_2.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_2.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/teams_4.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_4.nprint
Processing ../nd_data/color_corrected_generated_traffic_images/teams_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

The provided scripts handle this with color_processor.py (which thresholds fuzzy RGB values back to discrete anchors) and image_to_nprint.py (which maps pixels back into the exact schema using column_example.nprint). The advantage of their approach is that it’s fast, deterministic, and guarantees the output CSV has the exact shape downstream ML expects. The downside compared to my approach is that their pipeline relies purely on visual color bucketing and template matching without performing domain-level network validation, meaning it won't catch semantic errors like invalid checksums or impossible protocol state transitions.


---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [12]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Resolve real directory path dynamically
# ---------------------------------------------------------------
def resolve_directory(target_dir):
    candidates = [
        target_dir,
        os.path.join("..", target_dir),
        os.path.join(".", target_dir),
        os.path.abspath(target_dir)
    ]
    for path in candidates:
        if os.path.exists(path):
            files = glob.glob(os.path.join(path, "*.nprint*"))
            if len(files) > 0:
                return path
    raise FileNotFoundError(f"Could not locate files for directory: {target_dir}")


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path):
    files = glob.glob(os.path.join(example_path, "*.nprint*"))
    if not files:
        raise FileNotFoundError(f"No nprint files found in: {example_path}")

    first_file = files[0]
    df = pd.read_csv(first_file, low_memory=False)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.reindex(columns=base_cols, fill_value="0")
    df = df.iloc[:num_rows, :]
    df = df.map(safe_convert)

    # Pad with zeros if trace has fewer than num_rows packets
    if len(df) < num_rows:
        padding = pd.DataFrame(0, index=range(num_rows - len(df)), columns=base_cols)
        df = pd.concat([df, padding], ignore_index=True)

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    flat = df.values.flatten()
    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    files = glob.glob(os.path.join(directory, "*.nprint*"))
    for path in files:
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Locate directories and load DataFrames
# ---------------------------------------------------------------
real_path = resolve_directory("nd_data/real_nprints")
synth_path = resolve_directory("nd_data/generated_nprint")

print(f"Loading real traces from:      {real_path}")
print(f"Loading synthetic traces from: {synth_path}")

base_cols = get_original_columns(real_path)

df_synth = load_directory_as_df(synth_path, base_cols)
df_real  = load_directory_as_df(real_path, base_cols)

print("\nSynthetic DF shape:", df_synth.shape)
print(df_synth.head(3))

print("\nReal DF shape:", df_real.shape)
print(df_real.head(3))

Loading real traces from:      ../nd_data/real_nprints
Loading synthetic traces from: nd_data/generated_nprint

Synthetic DF shape: (100, 3265)
   pkt1_ipv4_ver_0  pkt1_ipv4_ver_1  pkt1_ipv4_ver_2  pkt1_ipv4_ver_3  \
0                1                0                0                0   
1                1                0                0                0   
2                0                0                0                0   

   pkt1_ipv4_hl_0  pkt1_ipv4_hl_1  pkt1_ipv4_hl_2  pkt1_ipv4_hl_3  \
0               0               0               0               1   
1               0               0               0               1   
2               0               0               0               0   

   pkt1_ipv4_tos_0  pkt1_ipv4_tos_1  ...  pkt3_icmp_roh_23  pkt3_icmp_roh_24  \
0                0                0  ...                -1                -1   
1                0                0  ...                -1                -1   
2                0                0  ...      

In [17]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------------
# 1. Align Application Labels and Separate Features
# ---------------------------------------------------------------
shared_labels = sorted(list(set(df_synth["label"]).intersection(set(df_real["label"]))))
print(f"Shared application classes ({len(shared_labels)}): {shared_labels}\n")

# Filter to shared classes
df_synth_filt = df_synth[df_synth["label"].isin(shared_labels)].reset_index(drop=True)
df_real_filt  = df_real[df_real["label"].isin(shared_labels)].reset_index(drop=True)

feature_cols = [c for c in df_synth_filt.columns if c != "label"]

# Fit label encoder
encoder = LabelEncoder()
encoder.fit(shared_labels)

X_synth = df_synth_filt[feature_cols].values
y_synth = encoder.transform(df_synth_filt["label"])

X_real = df_real_filt[feature_cols].values
y_real = encoder.transform(df_real_filt["label"])

# ---------------------------------------------------------------
# 2. Experiment A: Train on Synthetic -> Test on Real
# ---------------------------------------------------------------
clf_synth = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
clf_synth.fit(X_synth, y_synth)

y_pred_real = clf_synth.predict(X_real)
acc_synth_to_real = accuracy_score(y_real, y_pred_real)

print("=" * 65)
print("EXPERIMENT A: TRAIN ON SYNTHETIC -> TEST ON REAL")
print(f"Overall Accuracy: {acc_synth_to_real * 100:.2f}%")
print("=" * 65)
print("\nClassification Report:")
print(classification_report(y_real, y_pred_real, target_names=encoder.classes_, zero_division=0))

print("Confusion Matrix:")
cm_synth_to_real = pd.DataFrame(
    confusion_matrix(y_real, y_pred_real),
    index=[f"True: {c}" for c in encoder.classes_],
    columns=[f"Pred: {c}" for c in encoder.classes_]
)
print(cm_synth_to_real)

# ---------------------------------------------------------------
# 3. Experiment B: Train on Real -> Test on Synthetic
# ---------------------------------------------------------------
clf_real = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
clf_real.fit(X_real, y_real)

y_pred_synth = clf_real.predict(X_synth)
acc_real_to_synth = accuracy_score(y_synth, y_pred_synth)

print("\n" + "=" * 65)
print("EXPERIMENT B: TRAIN ON REAL -> TEST ON SYNTHETIC")
print(f"Overall Accuracy: {acc_real_to_synth * 100:.2f}%")
print("=" * 65)
print("\nClassification Report:")
print(classification_report(y_synth, y_pred_synth, target_names=encoder.classes_, zero_division=0))

# ---------------------------------------------------------------
# 4. Feature Importance Analysis
# ---------------------------------------------------------------
importances = clf_synth.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

print("\nTop 10 Most Discriminative Packet Features (from Synthetic Model):")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. {feature_cols[idx]:<30} (importance: {importances[idx]:.4f})")

Shared application classes (10): ['amazon', 'facebook', 'instagram', 'meet', 'netflix', 'teams', 'twitch', 'twitter', 'youtube', 'zoom']

EXPERIMENT A: TRAIN ON SYNTHETIC -> TEST ON REAL
Overall Accuracy: 28.00%

Classification Report:
              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        20
    facebook       0.00      0.00      0.00        20
   instagram       0.67      0.30      0.41        20
        meet       0.39      0.35      0.37        20
     netflix       0.80      0.20      0.32        20
       teams       0.27      0.95      0.42        20
      twitch       0.00      0.00      0.00        20
     twitter       0.22      1.00      0.36        20
     youtube       0.00      0.00      0.00        20
        zoom       0.00      0.00      0.00        20

    accuracy                           0.28       200
   macro avg       0.23      0.28      0.19       200
weighted avg       0.23      0.28      0.19       200

Confu

We evaluated traffic classification using the first three packets of each flow across 10 application classes with a Random Forest model. When trained on synthetic data and tested on real traces, the classifier achieved 28.00% accuracy, nearly triple the 10.00% random guessing baseline. It captured distinct signatures for applications like Twitter (100% recall), Teams (95% recall), and Netflix (80% precision), though it over-predicted these dominant classes. In the reverse direction (trained on real data and tested on synthetic data), the model achieved 35.00% accuracy, with perfect recall on Meet and YouTube. The top discriminative features were early packet header lengths and IP identification fields in packets 1 and 2. This confirms that visual diffusion captures broad flow geometry and early handshake characteristics, even though fine-grained bit artifacts limit complete transferability.